In [ ]:
import pandas as pd
import os
from pathlib import Path

folder_path = r'C:\Users\Jiri.Pillar\Desktop\pohyb25'
excel_files = list(Path(folder_path).glob('*.xlsx')) + list(Path(folder_path).glob('*.xls'))

dfs = [pd.read_excel(file) for file in excel_files]
df = pd.concat(dfs, ignore_index=True)
F_populace = df[['Rok', 'Číslo\nobce', 'Narození', 'Zemřelí', 'Přistě-\nhovalí','Vystě-\nhovalí','Přírůstek přirozený','Přírůstek migrační','Přírůstek celkový','Stav 31.12.']]
col_map = {
    'Rok': 'year',
    'Číslo\nobce': 'kod_obec',
    'Narození': 'narozeni',
    'Zemřelí': 'zemreli',
    'Přistě-\nhovalí': 'pristehovali',
    'Vystě-\nhovalí': 'vystehovali',
    'Přírůstek přirozený': 'prirustek_prirozeny',
    'Přírůstek migrační': 'prirustek_migracni',
    'Přírůstek celkový': 'prirustek_celkovy',
    'Stav 31.12.': 'populace',
}

# rename columns in both dataframes (works even if a dataframe lacks some keys)
F_populace.rename(columns=col_map, inplace=True)
# Convert all columns to int except 'název_obce'
for col in F_populace.columns:
    F_populace[col] = pd.to_numeric(F_populace[col], errors='coerce').astype('Int64')
    # filter F_populace to years 2013-2025 (inclusive)
F_populace = F_populace[F_populace['year'].between(2013, 2025)].reset_index(drop=True)
print(f"Filtered F_populace: {F_populace.shape} (years {F_populace['year'].min()}-{F_populace['year'].max()})")
path = Path(folder_path) / 'F_populace.csv'
F_populace.to_csv(path, index=False, encoding='utf-8')
print(f"Exported F_populace to: {path}")

In [28]:
import pandas as pd
from pathlib import Path

folder_path = Path(r'C:\Users\Jiri.Pillar\Desktop\pohyb25')

col_map = {
    'Rok': 'year',
    'Číslo\nobce': 'kod_obec',
    'Narození': 'narozeni',
    'Zemřelí': 'zemreli',
    'Přistě-\nhovalí': 'pristehovali',
    'Vystě-\nhovalí': 'vystehovali',
    'Přírůstek přirozený': 'prirustek_prirozeny',
    'Přírůstek migrační': 'prirustek_migracni',
    'Přírůstek celkový': 'prirustek_celkovy',
    'Stav 31.12.': 'populace',
}

files = [*folder_path.glob('*.xlsx'), *folder_path.glob('*.xls')]

F_populace = (
    pd.concat(
        (pd.read_excel(f, usecols=list(col_map)) for f in files),
        ignore_index=True,
    )
    .rename(columns=col_map)
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

F_populace = (
    F_populace[F_populace['year'].between(2013, 2025)]
    .sort_values(['kod_obec', 'year'])
    .reset_index(drop=True)
)

# duplicate 2025 as 2026 (placeholder)
dup_2026 = F_populace[F_populace['year'].eq(2025)].copy()
dup_2026['year'] = 2026

F_populace = (
    pd.concat([F_populace, dup_2026], ignore_index=True)
    .sort_values(['kod_obec', 'year'])
    .reset_index(drop=True)
)
print(f"F_populace: {F_populace.shape} (years {F_populace['year'].min()}-{F_populace['year'].max()})")

out = folder_path / 'F_populace.csv'
F_populace.to_csv(out, index=False, encoding='utf-8')
F_populace.to_parquet(folder_path / 'F_populace.parquet', index=False)
print(f"Exported to: {out}")
print(f"Exported to: {folder_path / 'F_populace.parquet'}")

F_populace: (87597, 10) (years 2013-2026)
Exported to: C:\Users\Jiri.Pillar\Desktop\pohyb25\F_populace.csv
Exported to: C:\Users\Jiri.Pillar\Desktop\pohyb25\F_populace.parquet
